# Inside the Agent Loop: Hands-On with CrewAI Before Agent Studio

**Agentic AI Training — Lab A | Day 1 | Module 2 | ~60 minutes**

---

## Overview

In this lab you will build a working **2-agent CrewAI crew** inside a Cloudera AI JupyterLab session using **Azure OpenAI** as the LLM backend.

Because Cloudera Agent Studio is built on top of CrewAI internally, completing this exercise first means you understand exactly what Agent Studio automates for you.

| | |
|---|---|
| **Environment** | Cloudera AI Workbench JupyterLab |
| **LLM** | Azure OpenAI (gpt-4o) |
| **CrewAI** | 1.14.1 |
| **Duration** | ~60 minutes |

---

## What you will build

A **News Research Crew** with two agents:

| Agent | Role | Tool | Output |
|---|---|---|---|
| Researcher | Senior Research Analyst | Web Search (Mock) | Numbered findings |
| Writer | Content Strategist | None | 3-paragraph brief |

---

## Pre-requisites — Set Cloudera AI Environment Variables

Before running any cell, add these four variables in **Cloudera AI Workbench → Project Settings → Advanced → Environment Variables**:

| Variable | Value | Sensitive? |
|---|---|---|
| `AZURE_OPENAI_API_KEY` | Your Azure API key | ✅ Yes — tick Secret |
| `AZURE_OPENAI_ENDPOINT` | `https://edu-cloudera-ai.openai.azure.com` | No |
| `AZURE_OPENAI_API_VERSION` | `2024-08-01-preview` | No |
| `AZURE_OPENAI_DEPLOYMENT` | `gpt-4o` | No |

> ⚠️ **After saving, stop and restart your Cloudera AI session** — environment variables are only injected at session start.

---

## Cell 0 — Install CrewAI

Run this cell first. It takes ~60 seconds.


> Dependency conflict warnings in the output are safe to ignore.

In [ ]:
%pip install crewai==1.14.1 --quiet

## Cell 1 — Verify CrewAI installation

Run this cell to confirm CrewAI imported correctly.

In [ ]:
import crewai
print(f"CrewAI version: {crewai.__version__}")  # should print 1.14.1

## Cell 2 — Verify Azure environment variables

Confirms Cloudera AI injected all four variables correctly.  
The API key is **masked** — only the last 4 characters are shown.

If any variable shows `MISSING`, go to **Cloudera AI project → Project Settings → Environment Variables**, add it, and restart your session.

In [ ]:
import os

required_vars = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_DEPLOYMENT",
]

print("Environment variable check:")
all_ok = True
for var in required_vars:
    val = os.environ.get(var, "")
    status = "OK" if val else "MISSING"
    if not val:
        all_ok = False
    display = "*" * 8 + val[-4:] if "KEY" in var else val
    print(f"  {status:8s}  {var}: {display}")

print()
if all_ok:
    print("✅ All variables set. Ready to proceed.")
else:
    print("❌ Some variables are missing. Fix them before continuing.")

## Cell 3 — Configure AzureDirectLLM

This custom class reads credentials from environment variables and calls Azure directly via `requests`.

**Why not use `LLM(model='azure/gpt-4o')` directly?**  
CrewAI 1.14.1 has a bug in its native Azure provider — it requires `azure-ai-inference` which is not installed in this CML runtime. The `AzureDirectLLM` class bypasses the broken routing entirely.

> 💡 **Note:** `AzureDirectLLM(model="azure/gpt-4o")` — the `model=` must be passed explicitly because CrewAI 1.14.1 uses Pydantic v2 which requires it at instantiation time.

In [ ]:
import os
import requests
from crewai.llms.base_llm import BaseLLM

# Read all credentials from CML environment variables — no hardcoded values
AZURE_ENDPOINT    = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
AZURE_DEPLOYMENT  = os.environ["AZURE_OPENAI_DEPLOYMENT"]
AZURE_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]
AZURE_API_KEY     = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_URL         = (
    f"{AZURE_ENDPOINT}/openai/deployments/{AZURE_DEPLOYMENT}"
    f"/chat/completions?api-version={AZURE_API_VERSION}"
)


class AzureDirectLLM(BaseLLM):
    """Calls Azure OpenAI directly via requests.
    Bypasses CrewAI 1.14.1 native Azure provider bug.
    """
    model: str = "azure/gpt-4o"  # required field in BaseLLM (Pydantic v2)

    def call(self, messages, tools=None, **kwargs):
        payload = {
            "messages": messages,
            "max_tokens": 1500,
            "temperature": 0.3,
        }
        response = requests.post(
            AZURE_URL,
            headers={"Content-Type": "application/json", "api-key": AZURE_API_KEY},
            json=payload,
            timeout=60,
        )
        response.raise_for_status()
        data = response.json()
        message = data["choices"][0]["message"]
        if message.get("tool_calls"):
            return message["tool_calls"]
        return message["content"]

    def supports_function_calling(self):
        return False  # forces ReAct text loop — simpler and more observable


# Instantiate — pass model= explicitly (Pydantic v2 requirement)
llm = AzureDirectLLM(model="azure/gpt-4o")
print("✅ LLM ready:", llm.model)

## Cell 4 — Define MockSearchTool

A simple tool that returns static search results.  
Used instead of `DuckDuckGoSearchRun` because `crewai-tools` has import conflicts on this CML runtime.

For this exercise the **content** of the results is not important — what matters is watching the agent **decide to call the tool** and **process the result**. That is the agent loop in action.

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from pydantic import BaseModel, Field


class SearchInput(BaseModel):
    query: str = Field(description="Search query string")


class MockSearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Searches for information on a given topic"
    args_schema: type[BaseModel] = SearchInput

    def _run(self, query: str) -> str:
        return (
            f"Search results for '{query}':\n"
            "1. Agentic AI is transforming enterprise data workflows in 2025.\n"
            "2. Multi-agent frameworks like CrewAI are widely adopted in production.\n"
            "3. Cloudera Agent Studio provides enterprise-grade agent orchestration.\n"
            "4. RAG remains the dominant pattern for grounding agent responses.\n"
            "5. MCP protocol is emerging as the standard for tool integration."
        )


search_tool = MockSearchTool()
print("✅ Tools ready:", search_tool.name)

## Cell 5 — Define Researcher and Writer agents

Each agent has three key fields:

| Field | Purpose |
|---|---|
| `role` | The agent's job title — shapes how it identifies itself |
| `goal` | What it is trying to achieve |
| `backstory` | The system prompt — shapes personality and behaviour |

Both agents receive `llm=llm` (the `AzureDirectLLM` from Cell 3).  
`max_iter=3` prevents infinite loops in a CML session with limited compute.

In [ ]:
researcher = Agent(
    role="Senior Research Analyst",
    goal="Find accurate, current information on {topic}",
    backstory=(
        "You are an expert analyst who uncovers clear, factual insights. "
        "You always cite key points and avoid speculation."
    ),
    tools=[search_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3,  # prevents infinite loops in CML sessions
)

writer = Agent(
    role="Content Strategist",
    goal="Turn research into a concise 3-paragraph professional brief",
    backstory=(
        "You are a skilled writer who transforms raw research into clear, "
        "professional briefings. You never invent facts."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agents defined:", researcher.role, "|", writer.role)

## Cell 6 — Define Tasks

A Task specifies **what to do**, the **expected output format**, and which **agent** is responsible.

- `{topic}` is a placeholder filled at runtime by the Crew
- `context=[research_task]` passes the Researcher's output directly to the Writer — this is the **task handoff** you will observe in the verbose trace

In [ ]:
research_task = Task(
    description=(
        "Search for recent developments on {topic}. "
        "Identify the 5 most important facts or trends. "
        "Present findings as a numbered list, one sentence per point."
    ),
    expected_output="A numbered list of 5 key findings about {topic}",
    agent=researcher,
)

write_task = Task(
    description=(
        "Using the researcher's findings, write a 3-paragraph professional "
        "briefing on {topic}. "
        "Para 1: overview. Para 2: key developments. Para 3: implications."
    ),
    expected_output="A polished 3-paragraph briefing on {topic}",
    agent=writer,
    context=[research_task],  # passes researcher output to writer
)

print("✅ Tasks defined.")

## Cell 7 — Assemble Crew and Run 🚀

The Crew ties everything together. `Process.sequential` runs tasks in order — research first, then writing.

### Watch the verbose output carefully — you will see:
1. **Researcher thinking** → reasoning before calling the tool
2. **Tool call** → the search query sent and results returned
3. **Researcher output** → numbered findings compiled from results
4. **Task handoff** → Writer receives Researcher output as context
5. **Writer output** → 3-paragraph brief composed without calling any tool

> 💡 Change the `topic` below to anything relevant to your project!

In [ ]:
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True,
)

try:
    result = crew.kickoff(
        inputs={"topic": "Agentic AI in enterprise data platforms 2025"}
    )
    print("\n" + "=" * 70)
    print(" FINAL BRIEFING")
    print("=" * 70)
    print(str(result))
except Exception as e:
    print(f"Error type : {type(e).__name__}")
    print(f"Error      : {e}")

---

## Reflection questions

Discuss with your neighbour or note your answers below:

**Q1.** Which part of the verbose output corresponds to the LLM reasoning step? How is it different from the tool call step?

**Q2.** The Writer produced a good output without calling any tool. When is an LLM alone sufficient vs when does an agent need a tool?

**Q3.** In Cloudera Agent Studio, where would you find the equivalent of the verbose trace you just read in the cell output?

---

## Bridge to Agent Studio

Every concept you just coded maps directly to Cloudera Agent Studio:

| What you wrote in CrewAI | Where it lives in Agent Studio |
|---|---|
| `Agent(role, goal, backstory)` | Agent config panel |
| `tools=[search_tool]` | Tool library → assign to agent |
| `Task(description, expected_output)` | Step / Action editor |
| `context=[research_task]` | Task dependency arrow on canvas |
| `Crew(agents, tasks, process)` | Pipeline canvas |
| `crew.kickoff(inputs={...})` | Run / Test button |
| `verbose=True` output | Trace viewer |
| `os.environ["AZURE_..."]` | Same CML project env vars — reused |

> **Key take-away:** The Azure environment variables you set in CML for this lab are the exact same variables Agent Studio uses. There is nothing to reconfigure when you move from notebook to platform.

---

## Bonus challenges (if time permits)

**Challenge A — Change the topic**  
Re-run Cell 7 with a topic relevant to your current project:
```python
result = crew.kickoff(inputs={"topic": "Cloudera Data Platform on Azure cost optimisation"})
```

**Challenge B — Add a third Reviewer agent**  
Add a `Fact Checker` agent after the Writer that verifies the brief contains no invented claims. Add it to the Crew's agents list and create a Task with `context=[write_task]`.

**Challenge C — Switch to hierarchical process**  
Change `Process.sequential` to `Process.hierarchical` and add `manager_llm=llm`. Watch how the delegation pattern changes in the verbose output.

---
*Cloudera   •  Agentic AI Training  •  Lab A  •  Cloudera AI + Azure OpenAI  •  CrewAI 1.14.1*